# 🔥 SAIL Digital Twin — AI-Assisted Stoichiometric Optimization
## Jupyter Notebook: EDA, Model Development & Training (Google Drive Edition)

**Project:** AI-Assisted Intelligent Stoichiometric Optimization in DFF Section Burners  
**Plant:** HDGL / CRM-III, SAIL Bokaro Steel Plant  
**Category:** ICC Technology Excellence Awards 2026 — Emerging Technology Adoption (Large)

---

### 📋 What This Notebook Does

1. **Mount Google Drive** — pull dataset from Drive, cache model artifacts back to Drive
2. **Skip-training shortcut** — if trained models exist in Drive, skip straight to export
3. **Data ingestion** — load canonical 3-year dataset (Periods A, B1, B2)
4. **Exploratory Data Analysis** — surface inefficiencies the AI addressed
5. **Feature engineering** — prep model-ready inputs
6. **Model development** — train XGBoost + MVPR ensemble (>98% accuracy)
7. **Export** — save artifacts to Google Drive for the Streamlit Digital Twin

### 🗓️ Canonical Timeline
| Period | FY | Status | Key Metric |
|---|---|---|---|
| **A** | FY 2023-24 | Legacy (pre-AI) | 16.644 kg propane/Ton |
| **B1** | FY 2024-25 | AI Year 1 | 13.741 kg/T · ₹3.23 Cr · 1,500.36 t CO₂ |
| **B2** | FY 2025-26 | AI Year 2 | 12.178 kg/T · ₹5.25 Cr · 2,429.61 t CO₂ |

## SECTION 1️⃣ — Environment Setup & Google Drive Mount

In [ ]:
# STEP 1.1 — Install dependencies (first-run on Colab)
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q pandas numpy scipy scikit-learn xgboost joblib \
                    matplotlib seaborn plotly openpyxl pyarrow streamlit

In [ ]:
# STEP 1.2 — Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
    r2_score, mean_absolute_percentage_error)
from sklearn.pipeline import Pipeline
import xgboost as xgb
import joblib

import os, warnings
from datetime import datetime
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
plt.rcParams['figure.figsize'] = (14, 6)
sns.set_style('whitegrid')

print(f'✅ Ready — {datetime.now().strftime("%Y-%m-%d %H:%M:%S")} · Colab: {IN_COLAB}')

In [ ]:
# STEP 1.3 — Mount Google Drive and define paths
# Change DRIVE_FOLDER to match your Google Drive folder path

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_FOLDER = '/content/drive/MyDrive/SAIL_Digital_Twin'
else:
    # Local fallback
    DRIVE_FOLDER = os.path.abspath('../drive')

DATASET_FOLDER   = os.path.join(DRIVE_FOLDER, 'dataset')
MODEL_FOLDER     = os.path.join(DRIVE_FOLDER, 'models')
DATA_OUT_FOLDER  = os.path.join(DRIVE_FOLDER, 'data_processed')

for f in [DRIVE_FOLDER, DATASET_FOLDER, MODEL_FOLDER, DATA_OUT_FOLDER]:
    os.makedirs(f, exist_ok=True)

print(f'📁 Drive:   {DRIVE_FOLDER}')
print(f'📊 Dataset: {DATASET_FOLDER}')
print(f'🧠 Models:  {MODEL_FOLDER}')
print(f'📈 Data:    {DATA_OUT_FOLDER}')

## SECTION 2️⃣ — Smart Training Shortcut

If trained models exist in Drive, skip training.

In [ ]:
REQUIRED_MODEL_FILES = [
    'scaler.pkl', 'xgb_models.pkl', 'mvpr_models.pkl',
    'feature_metadata.pkl', 'target_metadata.pkl',
    'performance_report.pkl', 'feature_cols.pkl', 'target_cols.pkl',
]
missing = [f for f in REQUIRED_MODEL_FILES if not os.path.exists(os.path.join(MODEL_FOLDER, f))]
MODEL_READY = len(missing) == 0

if MODEL_READY:
    print('🎯 Trained model found in Drive — will SKIP training.')
    print(f'   To force retrain: delete files in {MODEL_FOLDER}, or set FORCE_RETRAIN = True')
else:
    print(f'🆕 {len(missing)} artifacts missing — will train from scratch.')
    print(f'   Missing: {missing}')

FORCE_RETRAIN = False      # ← set True to force retrain
SKIP_TRAINING = MODEL_READY and not FORCE_RETRAIN
print(f'\n⚙️ SKIP_TRAINING = {SKIP_TRAINING}')

## SECTION 3️⃣ — Data Ingestion from Google Drive

In [ ]:
# STEP 3.1 — Locate dataset (CSV preferred, XLSX fallback)
csv_path  = os.path.join(DATASET_FOLDER, 'sail_master_5min.csv')
daily_csv = os.path.join(DATASET_FOLDER, 'sail_daily_summary.csv')
kpi_csv   = os.path.join(DATASET_FOLDER, 'sail_kpi_summary.csv')
xlsx_path = os.path.join(DATASET_FOLDER, 'SAIL_Master_Dataset.xlsx')

print('Scanning dataset folder...')
for f in [csv_path, daily_csv, kpi_csv, xlsx_path]:
    print(f'  {"✅" if os.path.exists(f) else "❌"} {os.path.basename(f)}')

if os.path.exists(csv_path):
    master = pd.read_csv(csv_path, parse_dates=['mydate'])
elif os.path.exists(xlsx_path):
    sheets = ['Period_A_FY23_24_Legacy', 'Period_B1_FY24_25_AI', 'Period_B2_FY25_26_AI']
    master = pd.concat([pd.read_excel(xlsx_path, sheet_name=s) for s in sheets], ignore_index=True)
    master['mydate'] = pd.to_datetime(master['mydate'])
else:
    raise FileNotFoundError(f'Dataset not found. Upload to {DATASET_FOLDER}')

print(f'\n✅ Master: {len(master):,} rows  ·  Periods: {sorted(master["Period"].unique())}')

In [ ]:
# STEP 3.2 — Load daily summary + KPI certification table
if os.path.exists(daily_csv):
    daily = pd.read_csv(daily_csv, parse_dates=['Date'])
elif os.path.exists(xlsx_path):
    daily = pd.read_excel(xlsx_path, sheet_name='Daily_Summary_All_3_Years')
    daily['Date'] = pd.to_datetime(daily['Date'])

if os.path.exists(kpi_csv):
    kpi = pd.read_csv(kpi_csv)
elif os.path.exists(xlsx_path):
    kpi = pd.read_excel(xlsx_path, sheet_name='KPI_Certification_Summary')

print(f'✅ Daily: {len(daily)} rows across {daily["FY"].nunique()} FYs')
print(f'✅ KPI Certification Summary:')
print(kpi[['Period','FY','Production_Tons','Specific_Propane_kg_per_Ton','Financial_Saving_Cr','CO2_Avoided_Tons']].to_string(index=False))

## SECTION 4️⃣ — Exploratory Data Analysis

In [ ]:
# STEP 4.1 — Running-state filter
def filter_running(df):
    return df[(df['coilthickness'] > 0) & (df['coilwidth'] > 0) & (df['linespeed'] > 0)].copy()

for p in ['A', 'B1', 'B2']:
    sub = master[master['Period'] == p]
    run = filter_running(sub)
    print(f'Period {p} ({sub["FY"].iloc[0]}): {len(run):,} running rows / {len(sub):,} total ({len(run)/len(sub)*100:.1f}% uptime)')

In [ ]:
# STEP 4.2 — Visualize 3-period transformation
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
colors = {'A': '#C00000', 'B1': '#2E74B5', 'B2': '#00B050'}
labels = {'A': 'A — Legacy (FY23-24)', 'B1': 'B1 — AI Y1 (FY24-25)', 'B2': 'B2 — AI Y2 (FY25-26)'}

for p in ['A', 'B1', 'B2']:
    sub = daily[daily['Period'] == p]
    axes[0,0].plot(sub['Date'], sub['Production (Tons)'], color=colors[p], alpha=0.6, label=labels[p], linewidth=0.8)
    axes[0,1].plot(sub['Date'], sub['Grand Total Power (kWh)'], color=colors[p], alpha=0.6, label=labels[p], linewidth=0.8)
axes[0,0].set_title('Daily Production (Tons)', fontweight='bold'); axes[0,0].legend(fontsize=8)
axes[0,1].set_title('Daily Grand Total Power (kWh)', fontweight='bold'); axes[0,1].legend(fontsize=8)

# Certified specific propane
periods_bar = ['A (FY23-24)', 'B1 (FY24-25)', 'B2 (FY25-26)']
propane_vals = [16.644, 13.741, 12.178]
bar_colors = ['#C00000', '#2E74B5', '#00B050']
bars = axes[1,0].bar(periods_bar, propane_vals, color=bar_colors)
for b, v in zip(bars, propane_vals):
    axes[1,0].text(b.get_x()+b.get_width()/2, v+0.2, f'{v:.3f}', ha='center', fontweight='bold')
axes[1,0].set_title('Specific Propane (kg/ton) — CERTIFIED', fontweight='bold')
axes[1,0].set_ylabel('kg/ton'); axes[1,0].set_ylim(0, 20)

# Certified savings
savings_vals = [3.23, 5.25, 8.48]
bars2 = axes[1,1].bar(['B1 Savings', 'B2 Savings', 'Cumulative'], savings_vals, color=['#2E74B5','#00B050','#D4A10F'])
for b, v in zip(bars2, savings_vals):
    axes[1,1].text(b.get_x()+b.get_width()/2, v+0.2, f'₹{v:.2f} Cr', ha='center', fontweight='bold')
axes[1,1].set_title('Financial Savings (₹ Cr) — CERTIFIED', fontweight='bold')
axes[1,1].set_ylabel('₹ Crores'); axes[1,1].set_ylim(0, 10)

plt.suptitle('SAIL Digital Twin — 3-Year Transformation Journey', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('eda_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## SECTION 5️⃣ — Feature Engineering

In [ ]:
# STEP 5.1 — Define features and targets
FEATURE_COLS = [
    'coilthickness', 'coilwidth', 'linespeed', 'pottemp',
    'vip11power', 'vip12power',
    'dffstrip_ACT', 'porrest', 'enloop', 'exloop',
]
TARGET_COLS = ['rthstrip_SP', 'rtsstrip_SP', 'rthzone_SP', 'rtszone_SP']

print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')
print(f'Targets ({len(TARGET_COLS)}): {TARGET_COLS}')

In [ ]:
# STEP 5.2 — Build training dataset from AI-deployed periods (B1 + B2)
if not SKIP_TRAINING:
    ai_data = master[master['AI_Deployed'] == True]
    train_df = filter_running(ai_data)[FEATURE_COLS + TARGET_COLS].dropna()
    for col in TARGET_COLS:
        q_low, q_high = train_df[col].quantile([0.01, 0.99])
        train_df = train_df[(train_df[col] >= q_low) & (train_df[col] <= q_high)]
    print(f'✅ Training data: {len(train_df):,} rows (B1 + B2 running state)')
else:
    print('⏭️ Skipping — loading pre-trained model later.')

## SECTION 6️⃣ — Model Training (runs only if SKIP_TRAINING = False)

In [ ]:
# STEP 6.1 — Split + scale
if not SKIP_TRAINING:
    X = train_df[FEATURE_COLS].values
    Y = train_df[TARGET_COLS].values
    thickness_bin = pd.cut(train_df['coilthickness'], bins=[0, 0.5, 0.8, 1.2, 3], labels=[0,1,2,3])
    X_train, X_test, Y_train, Y_test = train_test_split(
        X, Y, test_size=0.2, random_state=42, stratify=thickness_bin
    )
    scaler = StandardScaler().fit(X_train)
    X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)
    print(f'Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}')

In [ ]:
# STEP 6.2 — Train MVPR (interpretable polynomial)
if not SKIP_TRAINING:
    mvpr_models = {}
    for i, target in enumerate(TARGET_COLS):
        m = Pipeline([
            ('poly', PolynomialFeatures(degree=2, include_bias=False)),
            ('ridge', Ridge(alpha=1.0, random_state=42))
        ])
        m.fit(X_train_s, Y_train[:, i])
        mvpr_models[target] = m
    print('✅ MVPR trained for all 4 targets')

In [ ]:
# STEP 6.3 — Train XGBoost
if not SKIP_TRAINING:
    xgb_models = {}
    for i, target in enumerate(TARGET_COLS):
        m = xgb.XGBRegressor(
            n_estimators=400, max_depth=6, learning_rate=0.05,
            min_child_weight=10, subsample=0.9, colsample_bytree=0.9,
            reg_alpha=0.1, reg_lambda=1.0,
            random_state=42, n_jobs=-1, verbosity=0
        )
        m.fit(X_train_s, Y_train[:, i])
        xgb_models[target] = m
    print('✅ XGBoost trained for all 4 targets')

In [ ]:
# STEP 6.4 — Evaluate on held-out test set
if not SKIP_TRAINING:
    results = []
    for i, target in enumerate(TARGET_COLS):
        y_true = Y_test[:, i]
        y_mvpr = mvpr_models[target].predict(X_test_s)
        y_xgb  = xgb_models[target].predict(X_test_s)
        y_ens  = (y_mvpr + y_xgb) / 2
        for name, y_pred in [('MVPR', y_mvpr), ('XGBoost', y_xgb), ('Ensemble', y_ens)]:
            mae = mean_absolute_error(y_true, y_pred)
            rmse = np.sqrt(mean_squared_error(y_true, y_pred))
            r2 = r2_score(y_true, y_pred)
            mape = mean_absolute_percentage_error(y_true, y_pred) * 100
            acc = (1 - mape/100) * 100
            results.append({'Target': target, 'Model': name,
                            'MAE (°C)': round(mae,3), 'RMSE (°C)': round(rmse,3),
                            'R²': round(r2,4), 'MAPE (%)': round(mape,3), 'Accuracy (%)': round(acc,2)})
    results_df = pd.DataFrame(results)
    print(results_df.to_string(index=False))

In [ ]:
# STEP 6.5 — 5-fold CV (robustness check)
if not SKIP_TRAINING:
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    y_all = Y[:, TARGET_COLS.index('rthstrip_SP')]
    cv_scores = {'R²': [], 'MAE': [], 'Accuracy (%)': []}
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
        Xt, Xv = X[tr_idx], X[val_idx]
        yt, yv = y_all[tr_idx], y_all[val_idx]
        sc = StandardScaler().fit(Xt)
        m = xgb.XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                              min_child_weight=10, subsample=0.9, colsample_bytree=0.9,
                              random_state=42, n_jobs=-1, verbosity=0)
        m.fit(sc.transform(Xt), yt)
        preds = m.predict(sc.transform(Xv))
        r2, mae = r2_score(yv, preds), mean_absolute_error(yv, preds)
        mape = mean_absolute_percentage_error(yv, preds) * 100
        acc = (1 - mape/100) * 100
        cv_scores['R²'].append(r2); cv_scores['MAE'].append(mae); cv_scores['Accuracy (%)'].append(acc)
        print(f'  Fold {fold+1}: R²={r2:.4f} | MAE={mae:.3f}°C | Accuracy={acc:.2f}%')
    print(f'\n📊 CV Accuracy: {np.mean(cv_scores["Accuracy (%)"]):.2f}% ± {np.std(cv_scores["Accuracy (%)"]):.2f}%')
    if np.mean(cv_scores['Accuracy (%)']) >= 98:
        print('🏆 MODEL MEETS JURY-GRADE ACCURACY TARGET (>98%)')

## SECTION 7️⃣ — Export Artifacts to Google Drive

In [ ]:
# STEP 7.1 — Build feature & target metadata
if not SKIP_TRAINING:
    feature_desc = {
        'coilthickness': ('Coil Thickness', 'mm', 'Gauge of strip being processed'),
        'coilwidth':     ('Coil Width', 'mm', 'Lateral width of strip'),
        'linespeed':     ('Line Speed', 'mpm', 'Strip speed through furnace'),
        'pottemp':       ('Zinc Pot Temperature', '°C', 'Molten zinc bath temperature'),
        'vip11power':    ('VIP-11 Power', 'kW', 'Induction pre-heater 1'),
        'vip12power':    ('VIP-12 Power', 'kW', 'Induction pre-heater 2'),
        'dffstrip_ACT':  ('DFF Exit Strip Temp', '°C', 'Actual strip temperature exiting DFF'),
        'porrest':       ('POR Remaining', 'm', 'Pay-Off Reel remaining coil length'),
        'enloop':        ('Entry Loop', '—', 'Entry loop tension state'),
        'exloop':        ('Exit Loop', '—', 'Exit loop tension state'),
    }
    feature_metadata = {
        f: {'label': feature_desc[f][0], 'unit': feature_desc[f][1],
            'description': feature_desc[f][2],
            'min': float(train_df[f].quantile(0.01)),
            'max': float(train_df[f].quantile(0.99)),
            'default': float(train_df[f].median()),
            'std': float(train_df[f].std())}
        for f in FEATURE_COLS
    }

    target_desc = {
        'rthstrip_SP': ('RTH Strip Setpoint', '°C', 'Radiant Tube Heating strip setpoint'),
        'rtsstrip_SP': ('RTS Strip Setpoint', '°C', 'Radiant Tube Soaking strip setpoint'),
        'rthzone_SP':  ('RTH Zone Setpoint', '°C', 'RTH zone ambient setpoint'),
        'rtszone_SP':  ('RTS Zone Setpoint', '°C', 'RTS zone ambient setpoint'),
    }
    target_metadata = {
        t: {'label': target_desc[t][0], 'unit': target_desc[t][1],
            'description': target_desc[t][2],
            'min': float(train_df[t].quantile(0.01)),
            'max': float(train_df[t].quantile(0.99)),
            'mean': float(train_df[t].mean())}
        for t in TARGET_COLS
    }

In [ ]:
# STEP 7.2 — Performance report
if not SKIP_TRAINING:
    performance_report = {
        'training_samples':    int(len(X_train)),
        'test_samples':        int(len(X_test)),
        'n_features':          len(FEATURE_COLS),
        'n_targets':           len(TARGET_COLS),
        'cv_r2_mean':          float(np.mean(cv_scores['R²'])),
        'cv_r2_std':           float(np.std(cv_scores['R²'])),
        'cv_mae_mean':         float(np.mean(cv_scores['MAE'])),
        'cv_accuracy_mean':    float(np.mean(cv_scores['Accuracy (%)'])),
        'cv_accuracy_std':     float(np.std(cv_scores['Accuracy (%)'])),
        'test_metrics':        results_df.to_dict('records'),
        'feature_importance':  dict(zip(FEATURE_COLS, [float(x) for x in xgb_models['rthstrip_SP'].feature_importances_])),
        'training_timestamp':  datetime.now().isoformat(),
        'pipeline_version':    '1.1.0',
    }
    print(f'CV Accuracy: {performance_report["cv_accuracy_mean"]:.2f}%')

In [ ]:
# STEP 7.3 — Save artifacts to Google Drive
if not SKIP_TRAINING:
    artifacts = {
        'scaler.pkl':              scaler,
        'xgb_models.pkl':          xgb_models,
        'mvpr_models.pkl':         mvpr_models,
        'feature_metadata.pkl':    feature_metadata,
        'target_metadata.pkl':     target_metadata,
        'performance_report.pkl':  performance_report,
        'feature_cols.pkl':        FEATURE_COLS,
        'target_cols.pkl':         TARGET_COLS,
    }
    for name, obj in artifacts.items():
        joblib.dump(obj, os.path.join(MODEL_FOLDER, name))
    print(f'✅ Saved to {MODEL_FOLDER}:')
    for f in sorted(os.listdir(MODEL_FOLDER)):
        p = os.path.join(MODEL_FOLDER, f)
        print(f'   {f:28s}  {os.path.getsize(p)/1024:8.1f} KB')
else:
    print('Using pre-trained model from Drive.')

In [ ]:
# STEP 7.4 — Export dashboard-ready parquet files
def to_hourly(df):
    d = df.copy().set_index('mydate')
    numeric = d.select_dtypes(include=[np.number])
    return numeric.resample('h').mean().reset_index().dropna(how='all')

for p in ['A', 'B1', 'B2']:
    sub = master[master['Period'] == p]
    to_hourly(sub).to_parquet(os.path.join(DATA_OUT_FOLDER, f'hourly_period_{p}.parquet'), index=False)

    daily_sub = daily[daily['Period'] == p].copy().drop(columns=['Running Hours'], errors='ignore')
    daily_sub.to_parquet(os.path.join(DATA_OUT_FOLDER, f'daily_period_{p}.parquet'), index=False)

    run_sub = filter_running(sub)
    run_sub.sample(min(10000, len(run_sub)), random_state=42)\
           .to_parquet(os.path.join(DATA_OUT_FOLDER, f'sample_period_{p}.parquet'), index=False)

kpi.to_parquet(os.path.join(DATA_OUT_FOLDER, 'kpi_summary.parquet'), index=False)

print('✅ Dashboard data exported:')
for f in sorted(os.listdir(DATA_OUT_FOLDER)):
    p = os.path.join(DATA_OUT_FOLDER, f)
    print(f'   {f:30s}  {os.path.getsize(p)/(1024*1024):6.2f} MB')

## SECTION 8️⃣ — Final Summary

In [ ]:
print('=' * 70)
print('SAIL DIGITAL TWIN — READINESS CHECK')
print('=' * 70)

print('\n🎯 CERTIFIED CLAIMS:')
for _, row in kpi.iterrows():
    print(f'  {row["Period"]} ({row["FY"]}):')
    print(f'    Production:       {row["Production_Tons"]:>10,.0f} T')
    print(f'    Specific propane: {row["Specific_Propane_kg_per_Ton"]:>10.3f} kg/T')
    if pd.notna(row.get('Financial_Saving_Cr')):
        print(f'    Savings:          ₹{row["Financial_Saving_Cr"]:>9.2f} Cr')
        print(f'    CO₂ avoided:      {row["CO2_Avoided_Tons"]:>10,.2f} T')

if not SKIP_TRAINING:
    print(f'\n🧠 MODEL (freshly trained):')
    print(f'   CV Accuracy: {np.mean(cv_scores["Accuracy (%)"]):.2f}%')
    print(f'   CV MAE:      {np.mean(cv_scores["MAE"]):.2f}°C')
else:
    pr = joblib.load(os.path.join(MODEL_FOLDER, 'performance_report.pkl'))
    print(f'\n🧠 MODEL (from Drive cache):')
    print(f'   CV Accuracy: {pr["cv_accuracy_mean"]:.2f}%')
    print(f'   CV MAE:      {pr["cv_mae_mean"]:.2f}°C')
    print(f'   Trained:     {pr["training_timestamp"][:10]}')

print(f'\n📦 ARTIFACTS:')
print(f'   Models: {MODEL_FOLDER}')
print(f'   Data:   {DATA_OUT_FOLDER}')

print(f'\n🚀 READY FOR STREAMLIT DIGITAL TWIN DEPLOYMENT')
print('=' * 70)